# Notebook 5, Part 1: Mapping the 2005 MDR-TB Outbreak in Thailand

**Atlas source reading copy · local-server**

[Course guide](index.ipynb) · [Pinned original](https://github.com/PHI-Case-Studies/2005-MDR-TB-Outbreak-Thailand/blob/6d50f558cd39b40089cd3eea1fbc19425e5191d1/NB05-MDR-TB-Case-Study-Part-1.ipynb)



Use the source local/server environment for native GIS, shell, installed project or legacy dependencies; Lab provides a reading/editing copy. Follow notebook order to regenerate omitted outputs; optional unlicensed media are external continuations.

**Provenance:** `NB05-MDR-TB-Case-Study-Part-1.ipynb` at `6d50f558cd39b40089cd3eea1fbc19425e5191d1`; original SHA-256 `9d329fd538a6823d5e386588f8c725d17930c5827c51436f4028c01ba35a3b65`. Source-specific license/notices retained under notices; data and media have separate terms.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


In [ ]:
%run ./resources/library.py

In [ ]:
style_notebook()

Digital Case Study: Multidrug-Resistant Tuberculosis (MDR-TB) Outbreak - Revisiting the 2005 Outbreak Investigation in Thailand by John Oeltmann

# Notebook 5, Part 1: Mapping the 2005 MDR-TB Outbreak in Thailand

The Oeltmann et. al (2005) paper describes an outbreak of MDR-TB within a refugee camp in Thailand during 2005.  You may notice some similarities between the workflow pattern to map this outbreak with that of Dr. John Snow as he worked on the Cholera outbreak in London. You can find the 1854 Cholera Outbreak Case Study (Basic) [here](https://github.com/PHI-Case-Studies/1854-Cholera-Outbreak-London-Basic).

In [ ]:
show_pdf('resources/genotyping.pdf', 900,600)

For Notebooks 2 and 3, we will follow the workflow outlined in this [PDF file](https://github.com/PHI-Case-Studies/2005-MDR-TB-Outbreak-Thailand/blob/6d50f558cd39b40089cd3eea1fbc19425e5191d1/resources/Work_flow_diagram_08012017.pdf) below.

In [ ]:
show_pdf("./resources/Work_flow_diagram_08012017.pdf",800,500)

## Step 1. Outbreak Notification

You just received the **notification** below of an outbreak by email.

Please download the notification from your email ([PDF attachment](https://github.com/PHI-Case-Studies/2005-MDR-TB-Outbreak-Thailand/blob/6d50f558cd39b40089cd3eea1fbc19425e5191d1/resources/GDD.pdf)) using the PDF below. Please take time to read the contents of the attachment.

In [ ]:
show_pdf("./resources/GDD.pdf",800,500)

### Visualizing TB data from the World Health Organization

In [ ]:
import pandas as pd

pd.__version__

In [ ]:
import plotly.express as px
import plotly

plotly.__version__

In [ ]:
merged_df8 = pd.read_pickle('outputs/merged_df8.pickle')

merged_df8.head()

### TB mortality and GDP

In [ ]:
px.scatter(merged_df8, x="GDP-USD", y="e_mort_num", animation_frame="Year", \
           animation_group="Country", width=950, height=750, \
           size="e_pop_num", color="Region", hover_name="Country", \
           log_x=True, log_y=True, size_max=80, \
           range_x=[100,120000], range_y=[1,3000000])

### TB mortality and GDP per capita

In [ ]:
px.scatter(merged_df8, x="GDP-PC-USD", y="e_mort_num", animation_frame="Year", \
           animation_group="Country", width=950, height=750,\
           size="e_pop_num", color="Region", hover_name="Country", \
           log_x=True, log_y=True, size_max=80, \
           range_x=[100,120000], range_y=[1,3000000])

### TB mortality

In [ ]:
px.choropleth(merged_df8, locations="ISO_Alpha", color="e_mort_num", \
              hover_name="Country", animation_frame="Year", \
             color_continuous_scale=px.colors.sequential.OrRd)

To learn more about how these visualizations were developed, review Notebooks 2-4 (World TB data).

##  Step 2. Initial Line List of Cases Received

You then received a separate email with an attached [Excel Spreadsheet](resources/Thailand_cases_exercise_1st_spreadsheet_07232017.xls) that contains data about the initial cases. 

Now it's time to use your data science hacking skills. Using Python code, read this spreadsheet as a data frame to use within the Jupyter Notebooks environment.

In [ ]:
# Import the excel file and call it xls_file
xls_file = \
    pd.ExcelFile('resources/Thailand_cases_exercise_1st_spreadsheet_07232017.xls')
xls_file

In [ ]:
# View the excel file's sheet names
xls_file.sheet_names

In [ ]:
# Load the xls file's Sheet1 as a dataframe
df1 = xls_file.parse('TBDATA')

In [ ]:
# To view the column names with some data we can use the "head" function 
#    and indicate we want to display only 4 rows (the top 4).
# You can change the number 4 to something else, try it.
df1.head(4)

You can also read the Excel file [here](resources/Thailand_cases_exercise_1st_spreadsheet_07232017.xls) to check the outputs above.

> Question 1. What are the column names in the Excel spreadsheet? 

Answer Cell for Question #1.
Please type your answer below. Double-click to begin typing.


> Question 2. What is the name of the Sheet where the data are found?

> Question 3. How do you list the columns of the dataframe `df1`?

In [ ]:
# For Question #3, you may want to try typing the code here...
# Try: list(df1) or df1.columns.values


Let's "pickle" this data frame so we can retrieve it later.

In [ ]:
df1.to_pickle("outputs/df1.pickle")

## Step 3. Map 1: Mapping the Cases (data) in the Spreadsheet


Building on your knowledge of web mapping using `folium` from the Cholera outbreak case, let's use the same approach for mapping potential TB cases in the refugee camp.

In [ ]:
# use folium
import folium

folium.__version__

In [ ]:
#Store the coordinates of the refugee camp listed in the EOC email
CAMP_COORDINATES = (14.699859, 100.829019)

# create empty map zoomed in on Refugee camp
map1 = folium.Map(location=CAMP_COORDINATES, zoom_start=16)

# loop through data frame records
for each in df1.iterrows():
    # load record values into marker variables
    point=each[1]['COORDS']
    caseno=each[1]['CaseNo']
    # set up popup display string
    popup_string = "<b>Case No.:</b> "+caseno+"<br/>"+ \
                   "<b>Location:</b> "+point
    # create Folium popup
    popup=folium.Popup(popup_string, max_width='200')
    # add Folium marker to map
    folium.Marker(point.split(','), popup=popup).add_to(map1) 

# Let's display the map!                                       
map1

This being an old outbreak in a refugee camp, it is possible that the refugee camp does not exist anymore.

You can click on each marker in the map above to launch the pop-up that displays the case number. Note how you can include HTML formatting to make the popups look better.

## Step 4. Learning about Genotyping Terminology

The data received include a unique identifier for each case within the line listing. This may not always be present in real-life outbreak data, but is very helpful. We will be using this ID to join data from various sources, including molecular epidemiology data with genotypes of TB isolates. The CDC documentation on genotyping can help you to better understand the molecular epi data.

In a separate email, you receive Study IDs and genotypes within a 24-loci MIRU-VNTR coded system. You can read the genotyping documentation [here](https://github.com/PHI-Case-Studies/2005-MDR-TB-Outbreak-Thailand/blob/6d50f558cd39b40089cd3eea1fbc19425e5191d1/resources/genotypingterminology.pdf). Please take some time to understand the terminology used.

## Step 5. Genotyped Data Update from the Field!

You receive an email from the field with genotyping data in an [Excel spreadsheet](resources/Thailand_cases_exercise_2nd_spreadsheet_07232017.xls). This spreadsheet includes the unique ID for the patient and a 24-loci MIRU-VNTR string code as described in the CDC genotyping documentation. We will read this spreadsheet into a data frame in a similar manner to the case line listing data. 

Please run the Python code below to read the genotyping data.

In [ ]:
# Import the excel file and call it xls_file
xls_file2 = pd.ExcelFile('resources/Thailand_cases_exercise_2nd_spreadsheet_07232017.xls')
xls_file2

In [ ]:
# View the excel file's sheet names
xls_file2.sheet_names

In [ ]:
# Load the xls file's Sheet1 as a dataframe
df2 = xls_file2.parse('TBDATA2')

In [ ]:
# Let's display the dataframe
df2

In [ ]:
# Display the first four records of the dataframe df2
# Type the command below and press, Shift-Enter


Note that we have pre-assigned symbols in the last column to facilitate the visualization of different types of data on a map.

In [ ]:
# Let's pickle this dataframe and use it later
df2.to_pickle("outputs/df2.pickle")

## Step 6. Merging Two Data Sets

In the steps above, we you have created two dataframes:
1. `df1`: Line listing of cases 
2. `df2`: Cases with MIRU-VNTR (genotyping) string code

Next we will need to **`merge`** the case-line listing data with the genotyping data to create an analytic dataset with cases and their genotypes. In this step we will join the case-line listing data with the genotyping data and creating the analytic dataset, using the `CaseNo` field as the linking variable. 

Please run the Python code below, using `pandas merge` to merge these data:

In [ ]:
# Merge the dataframe with case-line listed data with the data frame with genotyping data...
df3 = pd.merge(df1, df2, left_on='CaseNo', right_on='CaseNo')
df3

In [ ]:
df3.to_pickle("outputs/df3.pickle")

## Step 7. Map 2: Mapping the Merged Data Sets

Please create a map of the TB cases, colored by MIRU ID from merged `df1` and `df2`. Let's use icons with different colors to distinguish records by MIRU IDs.

In [ ]:
# create 2nd map of the case data 
#Store the coordinates of the refugee camp listed in the EOC email
CAMP_COORDINATES = (14.699859, 100.829019)

# create empty map zoomed in on Refugee camp
map2 = folium.Map(location=CAMP_COORDINATES, zoom_start=16)

# create Python dictionary for icon colors
icon_color = {1:'red', 2:'yellow', 3:'blue', 4:'green', 5:'green'}

# loop through the data frame records
for each in df3.iterrows():
    # load record values in marker variables
    point=each[1]['COORDS']
    miru_id=each[1]['FAKEMIRUID']
    caseno=each[1]['CaseNo']
    icon=folium.Icon(color=icon_color[miru_id])
    # set up popup display string
    popup_string = "<b>Case No.:</b> "+caseno+"<br/>"+ \
                   "<b>Location:</b> "+point+"<br/>"+ \
                   "<b>MIRU ID:</b> "+str(miru_id)
    # create Folium popup
    popup=folium.Popup(popup_string, max_width='200')

    # add Folium marker to map
    folium.Marker(point.split(','), icon=icon, popup=popup).add_to(map2) 

map2


## Step 8. Incorporating Drug Resistance Type

Now that we have successfully joined the case line listing data with the genotyping data and created the analytic dataset, we will create a map of the genotyped cases and symbolize these cases by drug-resistance type.

We receive an email, our [third spreadsheet attached](resources/Thailand_cases_exercise_3rd_spreadsheet_07232017.xls), that includes a MIRU-ID and **classification of MIRU-ID by drug-resistance type**. We need to create a new field in the analytic dataset for DRTYPE and code each line-listed case by this type. 

Please run the Python code below to classify the MIRU-IDs by DRTYPE.

In [ ]:
# read the 3rd spreadsheet with MIRU-IDs and DRTYPE and create a data frame
# Import the excel file and call it xls_file
xls_file3 = pd.ExcelFile('resources/Thailand_cases_exercise_3rd_spreadsheet_07232017.xls')
xls_file3

In [ ]:
# View the excel file's sheet names
xls_file3.sheet_names

In [ ]:
# Load the xls file's Sheet1 as a dataframe
df4 = xls_file3.parse('TBDATA3')
df4
df4.to_pickle("outputs/df3.pickle")

In [ ]:
df4.head(4)

In [ ]:
# merge the dataframe with DRTYPE with the previous merge to form the analytic dataset
df5 = pd.merge(df4, df3, left_on=['CaseNo','FAKEMIRUVNTR','FAKEMIRUID'], \
               right_on=['CaseNo','FAKEMIRUVNTR','FAKEMIRUID'])

df5.head()

In [ ]:
df5.to_pickle('outputs/df5.pickle')

## Step 9. Map 3: Mapping the Analytic Genotyped Data Set

Next, we will create a map of the analytic dataset with genotyped TB cases, symbolized by drug-resistance type. We wish to see a map of the cases within the refugee camp and to examine the types of TB within the camp, including MDR-TB cases. Please run the code below to create the map of TB cases within the camp:


In [ ]:
# create 3rd map of the case data 
#Store the coordinates of the refugee camp listed in the EOC email
CAMP_COORDINATES = (14.699859, 100.829019)

# create empty map zoomed in on Refugee camp
map3 = folium.Map(location=CAMP_COORDINATES, zoom_start=16)
folium.TileLayer("openstreetmap").add_to(map3)

# create marker color Python dictionary
marker_color = {1:'red', \
               2:'yellow', \
               3:'blue', \
               4:'green', \
               5:'green'}

# loop through dataframe records
for each in df5.iterrows():
    # load record values into marker variables
    point=each[1]['COORDS']
    caseno=each[1]['CaseNo']
    miru_id=each[1]['FAKEMIRUID']
    drtype=each[1]['DRTYPE']
    # set up popup display string
    popup_string = "<b>Case No.:</b> "+caseno+"<br/>"+ \
                   "<b>Location:</b> "+point+"<br/>"+ \
                   "<b>MIRU ID:</b> "+str(miru_id)+"<br/>"+ \
                   "<b>Drug Resistance:</b> "+drtype
    # construct Folium popup
    popup=folium.Popup(popup_string, max_width='200')
    
    # Add the right marker to map depending on MIRU ID
    if miru_id == 1:
        folium.RegularPolygonMarker(point.split(','), \
            popup=popup, fill_color=marker_color[miru_id], \
            color='black', fill_opacity=1, weight=0.8,\
            number_of_sides=3, radius=10).add_to(map3)
    elif miru_id == 2:
        folium.RegularPolygonMarker(point.split(','), \
            popup=popup, fill_color=marker_color[miru_id], \
            color='black',fill_opacity=1, weight=0.8,\
            number_of_sides=4, radius=8).add_to(map3)
    elif miru_id == 3:
        folium.CircleMarker(point.split(','), \
            popup=popup, fill_color=marker_color[miru_id], \
            color='black',fill=True,fill_opacity=0.8, weight=0.7,\
            radius=7).add_to(map3)
    else:
        folium.CircleMarker(point.split(','), \
            popup=popup, fill_color=marker_color[miru_id], \
            color='black',fill=True, fill_opacity=1,weight=0.7,\
            radius=7).add_to(map3)

map3

You have now created a map similar to that found on page 1718 of the Oeltmann et. al 2005 paper.
[Source image continuation](https://github.com/PHI-Case-Studies/2005-MDR-TB-Outbreak-Thailand/blob/6d50f558cd39b40089cd3eea1fbc19425e5191d1/images/oeltmann_map.png)

## Final Map: With bells and whistles...

In [ ]:
import folium
import pandas as pd

from folium import plugins

df5 = pd.read_pickle('outputs/df5.pickle')

# create 3rd map of the case data 
#Store the coordinates of the refugee camp listed in the EOC email
CAMP_COORDINATES = (14.699859, 100.829019)

# create empty map zoomed in on Refugee camp
map3 = folium.Map(location=CAMP_COORDINATES, zoom_start=16)

folium.TileLayer("cartodbpositron").add_to(map3)

# we will add tile layer options
TileLayer1 = folium.TileLayer('openstreetmap')
TileLayer2 = folium.TileLayer('cartodbpositron')
TileLayer3 = folium.TileLayer('stamentoner')
TileLayer4 = folium.TileLayer('stamenterrain')

TileLayer1.layer_name = 'Open Street Map'
TileLayer2.layer_name = 'CartoDB Positron'
TileLayer3.layer_name = 'Stamen Toner'
TileLayer4.layer_name = 'Stamen Terrain'

TileLayer1.add_to(map3)
TileLayer2.add_to(map3)
TileLayer3.add_to(map3)
TileLayer4.add_to(map3)

fg = folium.FeatureGroup(name='All Markers')
map3.add_child(fg)

g1 = plugins.FeatureGroupSubGroup(fg, 'MDR-TB')
map3.add_child(g1)

g2 = plugins.FeatureGroupSubGroup(fg, 'Res to >1 drug')
map3.add_child(g2)

g3 = plugins.FeatureGroupSubGroup(fg, 'Pansusceptible')
map3.add_child(g3)

g4 = plugins.FeatureGroupSubGroup(fg, 'Unknown')
map3.add_child(g4)

# create marker color Python dictionary
marker_color = {1:'red', \
               2:'yellow', \
               3:'blue', \
               4:'green', \
               5:'green'}

# loop through dataframe records
for each in df5.iterrows():
    # load record values into marker variables
    point=each[1]['COORDS']
    caseno=each[1]['CaseNo']
    miru_id=each[1]['FAKEMIRUID']
    drtype=each[1]['DRTYPE']
    icon=folium.Icon(color=icon_color[miru_id])
    # set up popup display string
    popup_string = "<b>Case No.:</b> "+caseno+"<br/>"+ \
                   "<b>Location:</b> "+point+"<br/>"+ \
                   "<b>MIRU ID:</b> "+str(miru_id)+"<br/>"+ \
                   "<b>Drug Resistance:</b> "+drtype
    # construct Folium popup
    popup=folium.Popup(popup_string, max_width='200')
    
    # Add the right marker to map depending on MIRU ID
    if miru_id == 1:
        folium.RegularPolygonMarker(point.split(','), \
            popup=popup, fill_color=marker_color[miru_id], \
            color='black', fill_opacity=1, weight=0.8,\
            number_of_sides=3, radius=10).add_to(g1)
    elif miru_id == 2:
        folium.RegularPolygonMarker(point.split(','), \
            popup=popup, fill_color=marker_color[miru_id], \
            color='black',fill_opacity=1, weight=0.8,\
            number_of_sides=4, radius=8).add_to(g2)
    elif miru_id == 3:
        folium.CircleMarker(point.split(','), \
            popup=popup, fill_color=marker_color[miru_id], \
            color='black',fill=True,fill_opacity=0.8, weight=0.7,\
            radius=7).add_to(g3)
    else:
        folium.CircleMarker(point.split(','), \
            popup=popup, fill_color=marker_color[miru_id], \
            color='black',fill=True, fill_opacity=1,weight=0.7,\
            radius=7).add_to(g4)

# let's use the "Fullscreen" plugin
# add the button to the top right corner
plugins.Fullscreen(
    position='topright',
    title='Expand me',
    title_cancel='Exit me',
    force_separate_button=True
).add_to(map3)

# add the layer control
folium.LayerControl(collapsed=False).add_to(map3)

map3

You have been able to map the genotyped TB cases by drug resistance type within a refugee camp within Thailand. Please take some time to think about the questions at the end of this notebook and reflect upon the data requirements and purpose of this exercise. Thank you.

> Question 1: The TB case data provided in this exercise included the latitude and longitude coordinates gathered from GPS units. How critical are these geospatial data to completion of this exercise ?


## Congratulations!

## References


You have completed Notebook 5. Please proceed to Notebook 6.

CDC, 2017 Tuberculosis Genotyping: What is tuberculosis (TB) genotyping? CDC TB Fact Sheets 2017. URL: https://www.cdc.gov/tb/publications/factsheets/statistics/genotyping.htm

CDC, 2017 GENType: New Genotyping Terminology to Integrate 24-locus MIRU-VNTR. CDC TB Fact Sheets 2017. URL: https://www.cdc.gov/tb/publications/factsheets/statistics/genotypingterminology.pdf

CDC, 2017 A New Tool to Diagnose Tuberculosis: The Xpert MTB/RIF Assay. URL: https://www.cdc.gov/tb/publications/factsheets/pdf/xpertmtb-rifassayfactsheet_final.pdf

Oeltmann, J. E., Varma, J. K., Ortega, L., Liu, Y., O’Rourke, T., Cano, M., … Maloney, S. A. (2008). Multidrug-Resistant Tuberculosis  Outbreak among US-bound Hmong  Refugees, Thailand, 2005. Emerging Infectious Diseases, 14(11), 1715–1721. http://doi.org/10.3201/eid1411.071629

Shaw, N.S., et al. 2017. Transmission of Extensively Drug-Resistant Tuberculosis in South Africa. New England Journal of Medicine. January 19, 2017. 376:3.                                                  URL: http://www.nejm.org/doi/pdf/10.1056/NEJMoa1604544

Additional readings:
Reichmann, Lee B and Janice Hopkins Tanne. 2001. Timebomb: the global epidemic of multi-drug resistant tuberculosis. ISBN 0-07-135924-9. McGraw-Hill. URL: https://www.goodreads.com/book/show/1733578